# 02 · Data preparation
Follows handbook section 3.4. Fixed rules (`clean()`) run on all rows; anything learned from
the data (medians, modes, scaling, categories) is fitted on the **training set only**.
Output: `data/DSC_preprocessed_v2.joblib`, used by every model notebook.

In [1]:
import sys
from pathlib import Path

# find the project root (the folder that contains src/dsc_utils.py) and import the shared helpers
root = Path.cwd()
while not (root / "src" / "dsc_utils.py").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dsc_utils import *
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import joblib

raw_train, raw_score = load_raw()

## Step 1 · Clean training and scoring data with the same rules

In [2]:
status = raw_train[TARGET]
data = clean(raw_train)
score_ids = raw_score["finc_id"]
score_data = clean(raw_score)[data.columns]          # same columns, same order

print(data.shape, score_data.shape)
print("builtyear missing:", data["builtyear"].isna().mean().round(3))       # expect 0.0

(179446, 115) (76906, 115)
builtyear missing: 0.0


## Step 2 · Stratified train / validation / test split (64 / 16 / 20)

Note extra step: Stratify on label x region

In [3]:
indices = np.arange(len(data))

# stratify on label x region: the EPC scale differs per region (no G in Flanders, A+ almost only in Wallonia)
strata = status.astype(str) + "_" + data["region"]
counts = strata.value_counts()
strata = strata.where(strata.map(counts) >= 10, status.astype(str))   # tiny groups fall back to label only
print(counts.tail(6))                                                  # smallest label-region groups

indices_train, indices_test = train_test_split(indices, test_size=0.2, random_state=0, stratify=strata)
indices_train, indices_val = train_test_split(indices_train, test_size=0.2, random_state=0,
                                              stratify=strata.iloc[indices_train])
print(len(indices_train), len(indices_val), len(indices_test))

# check: label shares and label x region shares per set
print(pd.DataFrame({name: status.iloc[idx].value_counts(normalize=True).reindex(ORDER)
                    for name, idx in [("train", indices_train), ("val", indices_val), ("test", indices_test)]}).round(3))
print(pd.DataFrame({name: strata.iloc[idx].value_counts(normalize=True)
                    for name, idx in [("train", indices_train), ("val", indices_val), ("test", indices_test)]}).round(4).head(10))

A+_Wallonia    1243
C_Brussels     1129
B_Brussels      415
A_Brussels      166
A+_Flanders     146
G_Flanders        4
Name: count, dtype: int64
114844 28712 35890
           train    val   test
EPC label                     
A+         0.008  0.008  0.008
A          0.070  0.070  0.070
B          0.154  0.154  0.154
C          0.198  0.198  0.198
D          0.184  0.184  0.184
E          0.131  0.131  0.131
F          0.190  0.190  0.190
G          0.066  0.066  0.066
             train     val    test
F_Flanders  0.1368  0.1368  0.1368
C_Flanders  0.1337  0.1337  0.1337
B_Flanders  0.1200  0.1200  0.1200
D_Flanders  0.0984  0.0984  0.0984
D_Wallonia  0.0717  0.0717  0.0717
E_Flanders  0.0633  0.0632  0.0633
C_Wallonia  0.0578  0.0578  0.0578
A_Flanders  0.0573  0.0573  0.0573
E_Wallonia  0.0550  0.0550  0.0550
G_Wallonia  0.0484  0.0484  0.0484


## Step 3 · Manual imputation example (like Phone in the tutorial)

In [4]:
rooftype_mode = data["rooftype"].iloc[indices_train].mode()[0]       # mode from TRAIN only
print("Training mode:", rooftype_mode)
print("Missing before:", data["rooftype"].isna().sum(),
      "| after:", data["rooftype"].fillna(rooftype_mode).isna().sum())

Training mode: inclined
Missing before: 14739 | after: 0


3.2: Neighborhood Labels

In [5]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import KFold

coords = np.radians(data[["y_ll", "x_ll"]].to_numpy())
score_coords = np.radians(score_data[["y_ll", "x_ll"]].to_numpy())
y_num = status.map(TO_I).to_numpy()
poor = status.isin(["E", "F", "G"]).astype(float).to_numpy()

def neighbour_features(fit_idx, query_coords, k=20):
    out = []
    for target in [y_num, poor]:
        knn = KNeighborsRegressor(n_neighbors=k, metric="haversine", algorithm="ball_tree")
        knn.fit(coords[fit_idx], target[fit_idx])
        out.append(knn.predict(query_coords))
    return np.column_stack(out)

nb = np.full((len(data), 2), np.nan)
# training rows: out-of-fold, so a building never sees its own label
for fit_part, pred_part in KFold(5, shuffle=True, random_state=0).split(indices_train):
    nb[indices_train[pred_part]] = neighbour_features(indices_train[fit_part], coords[indices_train[pred_part]])
# validation and test rows: neighbours from all training rows
other = np.concatenate([indices_val, indices_test])
nb[other] = neighbour_features(indices_train, coords[other])

data["nb_mean_label"], data["nb_share_EFG"] = nb[:, 0], nb[:, 1]
nb_score = neighbour_features(indices_train, score_coords)
score_data["nb_mean_label"], score_data["nb_share_EFG"] = nb_score[:, 0], nb_score[:, 1]
print(data[["nb_mean_label", "nb_share_EFG"]].describe().round(2))

       nb_mean_label  nb_share_EFG
count      179446.00     179446.00
mean            3.96          0.39
std             0.73          0.18
min             1.05          0.00
25%             3.45          0.25
50%             3.90          0.35
75%             4.40          0.50
max             6.75          1.00


## Step 4 · Preprocessing pipeline (imputation, scaling, one-hot encoding)

In [6]:
cat_cols = data.select_dtypes(exclude="number").columns.tolist()
num_cols = data.select_dtypes(include="number").columns.tolist()
print(len(num_cols), "numeric |", len(cat_cols), "categorical")

preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=100))]), cat_cols),
])

x_train = preprocessor.fit_transform(data.iloc[indices_train])      # fit on TRAIN only
x_val   = preprocessor.transform(data.iloc[indices_val])
x_test  = preprocessor.transform(data.iloc[indices_test])
x_score = preprocessor.transform(score_data)
print("train", x_train.shape, "| val", x_val.shape, "| test", x_test.shape, "| scoring", x_score.shape)

48 numeric | 69 categorical
train (114844, 1618) | val (28712, 1618) | test (35890, 1618) | scoring (76906, 1618)


## Step 5 · Save everything the model notebooks need

In [7]:
joblib.dump({
    "x_train": x_train, "x_val": x_val, "x_test": x_test, "x_score": x_score,
    "y_train": status.iloc[indices_train].to_numpy(),
    "y_val": status.iloc[indices_val].to_numpy(),
    "y_test": status.iloc[indices_test].to_numpy(),
    "indices_train": indices_train, "indices_val": indices_val, "indices_test": indices_test,
    "feature_names": preprocessor.get_feature_names_out(),
    "preprocessor": preprocessor,
    "data": data, "score_data": score_data, "score_ids": score_ids.to_numpy(),
}, PREP_FILE)
print("Saved to", PREP_FILE)

Saved to /Users/trangtnguyen/Desktop/DSC_Groupe22/data/DSC_preprocessed_v2.joblib
